In [9]:
"""
Etosha National Park — Competitive Lotka-Volterra Food Web Simulation
======================================================================
Reads:  immc_-_species_final__3_.csv
        immc_-_foodweb_final__5_.csv
 
Produces two plots:
    etosha_clv_80yr.png    — full 80-year simulation
    etosha_clv_2yr.png     — first 2 years zoomed in (monthly detail)
 
Run:
    python3 etosha_clv_simulation.py
 
Dependencies:
    numpy, pandas, scipy, matplotlib
"""
 
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from collections import defaultdict
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
 
 
# ═════════════════════════════════════════════════════════════════════════════
# 1.  LOAD DATA
# ═════════════════════════════════════════════════════════════════════════════
 
species_df = pd.read_csv('species_sens_final.csv') 
foodweb_df = pd.read_csv('foodweb.csv')
 
CODES  = list(species_df['code'])
NAMES  = list(species_df['species'])
TYPES  = list(species_df['type'])
MASSES = np.array(species_df['body_mass_kg'], dtype=float)
 
N0 = species_df['initial_pop'].fillna(0).to_numpy(dtype=float).copy()
N0[CODES.index('PLANTS')] = 800_000.0   # start below K so seasonal rise is visible
N0[CODES.index('BUGS')]   = 400_000.0
 
n   = len(CODES)
idx = {c: i for i, c in enumerate(CODES)}
 
# Build prey list per predator from food web CSV
pp = defaultdict(list)
for _, row in foodweb_df.iterrows():
    pp[row['pred_code']].append(row['prey_code'])
 
 
# ═════════════════════════════════════════════════════════════════════════════
# 2.  ECOLOGICAL PARAMETERS
# ═════════════════════════════════════════════════════════════════════════════
 
# Carrying capacities (animals: approximately 1.5–2× Etosha census counts)
K_dict = {
    'PLANTS': 2_000_000, 'BUGS':   5_000_000,
    'AEPMEL':     8_000, 'ALCBUS':     1_000, 'CONTAU':    10_000,
    'EQUBUR':    25_000, 'GIRCAM':     5_000, 'LOXAFR':     5_000,
    'MADKIR':     8_000, 'PHAACT':     8_000, 'PROCAP':    30_000,
    'TAUORY':     5_000, 'ORYXOR':     8_000, 'BRHINO':     1_500,
    'WRHINO':     2_500, 'ANTELO':    25_000, 'PANGOL':    15_000,
    'AARDVA':    25_000, 'PANLEO':       800, 'PANPAR':       700,
    'CROCRO':       800, 'ACIJUB':       400, 'CARCAR':       300,
    'LEPSER':       600,
}
K_arr = np.array([K_dict.get(c, 5_000) for c in CODES], dtype=float)
 
# Allometric intrinsic growth rates:  r_i = r0 * M_i^(-1/4)
# Smaller animals have faster life histories (higher birth and death rates).
r0 = 0.8
r  = r0 * MASSES**(-0.25)
r[idx['PLANTS']] = 3.5    # grass net primary productivity (yr^-1)
r[idx['BUGS']]   = 10.0   # insect colony turnover (yr^-1)
 
IP = idx['PLANTS']
IB = idx['BUGS']
PRODUCER_IDX = {IP, IB}
 
 
# ═════════════════════════════════════════════════════════════════════════════
# 3.  COMPETITIVE LV ALPHA MATRIX
# ═════════════════════════════════════════════════════════════════════════════
# The CLV equation for every animal species i is:
#
#   dN_i/dt = r_i * N_i * (1 - (N_i + Σ_j α_ij * N_j) / K_i) * s(t)
#
# α_ij encodes all food web interactions:
#   α_ii = 1                         intraspecific (always)
#   α[prey_j, pred_i] = C_sup / n    predator suppresses prey  (> 0)
#   α[pred_i, prey_j] = -benefit     prey relieves predator     (< 0)
#   α_ij = 0                         no interaction
#
# C_SUP = 1.2 : total suppression a predator exerts on prey, split equally
#               among all predators that eat that prey species and all prey
#               species the predator eats.
#
# BENEFIT_FRAC = 0.30 : predator's total negative pressure from all prey
#               is at most 30% of its own K at initial populations.
#               This prevents predator populations from growing infinitely
#               when prey is abundant.
 
C_SUP        = 1.2
BENEFIT_FRAC = 0.30
 
alpha = np.zeros((n, n))
for i in range(n):
    alpha[i, i] = 1.0   # intraspecific competition
 
# Count predators eating each prey (to split C_SUP fairly)
n_predators_on = defaultdict(int)
for pred, prey_list in pp.items():
    for prey in prey_list:
        if idx[prey] not in PRODUCER_IDX:
            n_predators_on[idx[prey]] += 1
 
for pred, prey_list in pp.items():
    i      = idx[pred]
    n_prey = len(prey_list)
    animal_prey = [p for p in prey_list if idx[p] not in PRODUCER_IDX]
 
    for prey in prey_list:
        j = idx[prey]
        if j in PRODUCER_IDX:
            continue
        # α[prey, pred] > 0 : predator adds to prey's effective crowding
        n_preds_on_j = max(n_predators_on[j], 1)
        alpha[j, i] = (C_SUP / n_preds_on_j) / max(n_prey, 1)
 
    if animal_prey:
        # α[pred, prey] < 0 : prey reduces predator's effective crowding
        # Total benefit = -BENEFIT_FRAC * K_i, weighted by prey N0
        total_N_prey = sum(N0[idx[p]] for p in animal_prey)
        for prey in animal_prey:
            j      = idx[prey]
            weight = N0[j] / max(total_N_prey, 1.0)
            alpha[i, j] = -weight * BENEFIT_FRAC * K_arr[i] / max(N0[j], 1.0)
 
 
# ═════════════════════════════════════════════════════════════════════════════
# 4.  HERBIVORE CONSUMPTION RATES ON PRODUCERS
# ═════════════════════════════════════════════════════════════════════════════
# Producers (Plants, Bugs) use a classic logistic resource equation:
#
#   dP/dt = r_P * P * (1 - P/K_P) * s(t)  -  Σ_i c_i * N_i * P
#
# where c_i is the per-capita consumption rate of herbivore i on plant P.
#
# Calibration of c_i:
#   c_i * N_plant = r_i / n_plant_prey
#   => c_i = r_i / (n_plant_prey * K_plant/2)
#
#   (K_plant/2 is the midpoint of the logistic — where plant growth is fastest.
#    At this density, each herbivore's consumption equals a share of its own
#    birth rate, which is ecologically consistent.)
#
# Safety cap: sum_i c_i * K_i <= 70% of plant max growth rate.
# This ensures producers are never consumed faster than they can regrow
# even when all herbivores are at carrying capacity.
 
herb_rate = {}   # (herbivore_idx, producer_idx) -> consumption coefficient
 
for prod_code, prod_idx in [('PLANTS', IP), ('BUGS', IB)]:
    K_eq = K_arr[prod_idx] / 2.0
 
    grazers = [(idx[pred], prod_idx)
               for pred, prey_list in pp.items()
               for prey in prey_list if prey == prod_code]
 
    # Count plant-type prey per species (to split consumption evenly)
    n_plant_prey_of = {
        idx[pred]: sum(1 for pr in prey_list
                       if TYPES[idx[pr]] == 'producer' or pr == 'BUGS')
        for pred, prey_list in pp.items()
    }
 
    for i, j in grazers:
        n_pp = max(n_plant_prey_of.get(i, 1), 1)
        herb_rate[(i, j)] = r[i] / (n_pp * K_eq)
 
    # Apply safety cap
    max_herb   = sum(herb_rate[(i, j)] * K_arr[i] for i, j in grazers)
    max_growth = 0.70 * r[prod_idx] * K_arr[prod_idx] / 4.0
    if max_herb > max_growth and max_herb > 0:
        scale = max_growth / max_herb
        for i, j in grazers:
            herb_rate[(i, j)] *= scale
 
 
# ═════════════════════════════════════════════════════════════════════════════
# 5.  SEASONAL FORCING
# ═════════════════════════════════════════════════════════════════════════════
# s(t) = 1 + 0.4 * sin(2π t/12 - π/2)
#
# t is in months. The -π/2 phase shift moves the peak to February,
# matching Etosha's wet season. Amplitude ±40%:
#   Wet season (Feb): s = 1.4  → 40% above baseline growth
#   Dry season (Aug): s = 0.6  → 40% below baseline growth
#
# Applied to producers and prey. Not applied to carnivores directly —
# their reproductive rate is encoded in α (prey availability), not season.
 
def season(t):
    return 1.0 + 0.40 * np.sin(2.0 * np.pi * t / 12.0 - np.pi / 2.0)
 
 
# ═════════════════════════════════════════════════════════════════════════════
# 6.  ODE SYSTEM
# ═════════════════════════════════════════════════════════════════════════════
 
def food_web_ode(t, N):
    N  = np.maximum(N, 0.0)
    dN = np.zeros(n)
    s  = season(t)
 
    # ── Producers ─────────────────────────────────────────────────────────
    # dP/dt = r * P * (1 - P/K) * s(t)  -  Σ_i c_i * N_i * P
    for pi in PRODUCER_IDX:
        eaten = sum(herb_rate.get((i, pi), 0.0) * N[i] * N[pi]
                    for i in range(n))
        dN[pi] = r[pi] * N[pi] * (1.0 - N[pi] / K_arr[pi]) * s - eaten
 
    # ── Animals: Competitive LV ───────────────────────────────────────────
    # dN_i/dt = r_i * N_i * (1 - (N_i + Σ_j α_ij*N_j) / K_i) * [s(t)]
    for i in range(n):
        if i in PRODUCER_IDX:
            continue
        pressure = N[i] + sum(alpha[i, j] * N[j]
                              for j in range(n)
                              if j != i and alpha[i, j] != 0.0)
        growth = r[i] * N[i] * (1.0 - pressure / K_arr[i])
        # Seasonal forcing on prey; carnivores regulated by prey availability
        if TYPES[i] in ('herbivore', 'insectivore', 'omnivore'):
            dN[i] = growth * s
        else:
            dN[i] = growth
 
    return dN
 
 
# ═════════════════════════════════════════════════════════════════════════════
# 7.  SOLVE
# ═════════════════════════════════════════════════════════════════════════════
 
T_YEARS  = 80
T_MONTHS = T_YEARS * 12
t_eval   = np.linspace(0, T_MONTHS, T_MONTHS * 20)   # 20 pts/month
 
print("Solving Competitive Lotka-Volterra system...")
print(f"  Species: {n},  Duration: {T_YEARS} yr,  Steps: {len(t_eval):,}")
 
sol = solve_ivp(
    food_web_ode,
    (0, T_MONTHS),
    N0.copy(),
    method='RK45',
    t_eval=t_eval,
    rtol=1e-6,
    atol=1e-8,
    max_step=0.2,
)
print(f"  Solver: {'OK' if sol.success else sol.message}")
 
t_yr = sol.t / 12.0
Y    = np.maximum(sol.y, 0.0)
 
# ── Results table ──────────────────────────────────────────────────────────
t40 = len(t_eval) // 2
print(f"\n{'Species':30s}  {'Yr 0':>10s}  {'Yr 40':>10s}  {'Yr 80':>10s}")
print("-" * 66)
for i in range(n):
    row = f"  {NAMES[i]:28s}  {Y[i,0]:>10,.0f}  {Y[i,t40]:>10,.0f}  {Y[i,-1]:>10,.0f}"
    if Y[i, -1] < 1:
        row += "  EXTINCT"
    print(row)
survivors = sum(1 for i in range(n) if Y[i, -1] >= 1)
print(f"\nSurvivors at year 80: {survivors}/{n}")
 
 
# ═════════════════════════════════════════════════════════════════════════════
# 8.  PLOT SETTINGS
# ═════════════════════════════════════════════════════════════════════════════
 
BG   = "#ffffff"
CELL = "#ffffff"
GRID = '#21262d'
 
GROUPS = {
    'Producers': {
        'codes':  ['PLANTS', 'BUGS'],
        'colors': ['#2e7d32', '#66bb6a'],
    },
    'Megaherbivores': {
        'codes':  ['LOXAFR', 'GIRCAM', 'BRHINO', 'WRHINO'],
        'colors': ['#1565c0', '#42a5f5', '#0288d1', '#4dd0e1'],
    },
    'Large Herbivores': {
        'codes':  ['EQUBUR', 'CONTAU', 'TAUORY', 'ANTELO', 'ORYXOR', 'ALCBUS'],
        'colors': ['#6a1b9a', '#ab47bc', '#ce93d8', '#e040fb', '#ba68c8', '#7b1fa2'],
    },
    'Small Herbivores': {
        'codes':  ['AEPMEL', 'MADKIR', 'PROCAP', 'PHAACT'],
        'colors': ['#e65100', '#ff7043', '#ffb74d', '#bcaaa4'],
    },
    'Insectivores': {
        'codes':  ['PANGOL', 'AARDVA'],
        'colors': ['#bf360c', '#ff8a65'],
    },
    'Carnivores': {
        'codes':  ['PANLEO', 'PANPAR', 'CROCRO', 'ACIJUB', 'CARCAR', 'LEPSER'],
        'colors': ['#b71c1c', '#e53935', '#ef9a9a', '#ff8f00', '#ffd600', '#4dd0e1'],
    },
}
 
def style_ax(ax, gname, gdata, t_data, y_data, xlim,
             xticks=None, xticklabels=None, xlabel='Year'):
    ax.set_facecolor(CELL)
    for sp in ax.spines.values():
        sp.set_edgecolor('#bbbbbb')
    for k, code in enumerate(gdata['codes']):
        i   = idx[code]
        col = gdata['colors'][k % len(gdata['colors'])]
        ax.plot(t_data, y_data[i], color=col, lw=2.0, label=NAMES[i], alpha=0.93)
    ax.set_title(gname, color='black', fontsize=15, fontweight='bold', pad=8)
    ax.set_xlabel(xlabel, color='black', fontsize=12)
    ax.set_ylabel('Population', color='black', fontsize=12)
    ax.tick_params(colors='black', labelsize=11)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _:
        f'{v/1e6:.1f}M' if v >= 1e6 else
        f'{v/1e3:.0f}k' if v >= 1_000 else f'{v:.0f}'))
    if xticks is not None:
        ax.set_xticks(xticks)
        ax.set_xticklabels(xticklabels, fontsize=10, color='black', rotation=45)
    ax.legend(fontsize=11, framealpha=0.9, labelcolor='black',
              facecolor='white', edgecolor='#bbbbbb', loc='upper right', ncol=1)
    ax.grid(True, color=GRID, lw=0.6, alpha=0.8)
    ax.set_xlim(*xlim)

 
 
# ═════════════════════════════════════════════════════════════════════════════
# 9.  PLOT A — 80-YEAR OVERVIEW
# ═════════════════════════════════════════════════════════════════════════════
 
fig = plt.figure(figsize=(22, 26), facecolor=BG)
fig.suptitle(
    'Etosha National Park — Competitive Lotka-Volterra Food Web (80 Years)\n'
    'α competition matrix · Allometric rates · Seasonal forcing',
    fontsize=15, color='white', y=0.985, fontweight='bold'
)
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.44, wspace=0.30,
                       left=0.07, right=0.97, top=0.94, bottom=0.04)
 
for ax_idx, (gname, gdata) in enumerate(GROUPS.items()):
    ax = fig.add_subplot(gs[ax_idx // 2, ax_idx % 2])
    ax.axvspan(60, 80, color='white', alpha=0.04)
    ax.axvline(60, color='white', alpha=0.15, lw=0.8, linestyle='--')
    style_ax(ax, gname, gdata, t_yr, Y, (0, T_YEARS))
 
plt.savefig('etosha_clv_80yr.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print("\nSaved: etosha_clv_80yr.png")
 
 
# ═════════════════════════════════════════════════════════════════════════════
# 10. PLOT B — 2-YEAR ZOOM (monthly detail)
# ═════════════════════════════════════════════════════════════════════════════
 
# Solve again at very fine resolution over just 2 years
T2_MONTHS = 36
t2_eval   = np.linspace(0, T2_MONTHS, T2_MONTHS * 30)   # 30 pts/month ≈ daily
 
sol2 = solve_ivp(
    food_web_ode,
    (0, T2_MONTHS),
    N0.copy(),
    method='RK45',
    t_eval=t2_eval,
    rtol=1e-8,
    atol=1e-10,
    max_step=0.05,
)
t2_yr = sol2.t / 12.0
Y2    = np.maximum(sol2.y, 0.0)
 
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
 
# Month ticks: one per month, label every 3rd
tick_years  = np.arange(0, T2_MONTHS / 12.0 + 0.1, 0.5)   # every 6 months
tick_labels = [f'{t:.1f}' for t in tick_years]

 
fig2 = plt.figure(figsize=(22, 26), facecolor=BG)
fig2.suptitle(
    'Etosha Wildlife Competitive Lotka-Volterra Results',
    fontsize=20, color='black', y=0.99, fontweight='bold'
)
gs2 = gridspec.GridSpec(3, 2, figure=fig2, hspace=0.18, wspace=0.25,
                        left=0.08, right=0.97, top=0.95, bottom=0.08)

for ax_idx, (gname, gdata) in enumerate(GROUPS.items()):
    ax = fig2.add_subplot(gs2[ax_idx // 2, ax_idx % 2])
 
    # Year divider
    ax.axvline(1.0, color='white', alpha=0.25, lw=1.0, linestyle='--')
    ax.text(0.75, 0.97, 'Year 2', transform=ax.transAxes,
        color='black', fontsize=11, ha='center', va='top')

 
    style_ax(ax, gname, gdata, t2_yr, Y2,
         xlim=(0, T2_MONTHS / 12.0),
         xticks=tick_years,
         xticklabels=tick_labels,
         xlabel='Year')

 
 
plt.savefig('etosha_clv_2yr.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print("Saved: etosha_clv_2yr.png")

Solving Competitive Lotka-Volterra system...
  Species: 24,  Duration: 80 yr,  Steps: 19,200
  Solver: OK

Species                               Yr 0       Yr 40       Yr 80
------------------------------------------------------------------
  Plants                           800,000   1,964,746   1,964,804
  Ants/Termites                    400,000   4,995,685   4,995,688
  Cheetah                              500       3,027       3,027
  Black-Faced Impala                   500       7,299       7,299
  Hartebeest                           500         307         307
  Caracal                              500       3,319       3,319
  Wildebeest                           500       9,299       9,299
  Spotted/Brown Hyena                  500       6,811       6,811
  Plains Zebra                         500      24,307      24,307
  Giraffe                              500       4,355       4,355
  Serval                               500       6,639       6,639
  African Elephant    